# Charcha Weekly Profit Maximization with AMPL in Python
## AMPL Modeling for Problem 2

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Charcha Weekly Profit Maximization

Charcha buys raw material at $6/lb (unlimited supply).

**Raw material processing:**
- 1 lb → 2 oz Input 1 (2h, $2 cost)
- 1 lb → 3 oz Input 2 (2h, $4 cost)

**Production processes:**
- Process 1: 2h, 2 oz Input 1, 1 oz Input 2, $1 cost → 1 oz Product A + 1 oz waste
- Process 2: 3h, 1 oz Input 1, 2 oz Input 2, $8 cost → 1 oz Product B + 0.8 oz waste

**Waste products:**
- Dump in river (max 1,000 oz/week)
- Product C: $4 cost, sells $11, 1h, 2 oz Input 1, 0.8 oz waste → 1 oz
- Product D: $5 cost, sells $7, 1h, 2 oz Input 2, 1.2 oz waste → 1 oz

**Sales:**
- Product A: $18/oz, max 5,000 oz/week
- Product B: $24/oz, max 5,000 oz/week
- Products C, D: unlimited demand
- Processing time: 6,000 h/week

Maximize weekly profit.

In [4]:
@timed
def solve_charcha(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var mp1 >= 0;
        var mp2 >= 0;
        var p1 >= 0;
        var p2 >= 0;
        var prodC >= 0;
        var prodD >= 0;
        var waste_river >= 0;

        maximize Profit:
            18 * p1 + 24 * p2 + 11 * prodC + 7 * prodD
            - 6 * (mp1 + mp2)
            - 2 * mp1 - 4 * mp2
            - 1 * p1 - 8 * p2
            - 4 * prodC - 5 * prodD;

        subject to Input1Balance:
            2 * mp1 >= 2 * p1 + 1 * p2 + 2 * prodC;

        subject to Input2Balance:
            3 * mp2 >= 1 * p1 + 2 * p2 + 2 * prodD;

        subject to WasteBalance:
            1 * p1 + 0.8 * p2 = waste_river + 0.8 * prodC + 1.2 * prodD;

        subject to RiverLimit:
            waste_river <= 1000;

        subject to DemandA:
            p1 <= 5000;

        subject to DemandB:
            p2 <= 5000;

        subject to ProcessingTime:
            2 * mp1 + 2 * mp2 + 2 * p1 + 3 * p2 + 1 * prodC + 1 * prodD <= 6000;
        '''
    )
    ampl.solve(solver=solver)

    variables = ["mp1", "mp2", "p1", "p2", "prodC", "prodD", "waste_river"]
    solution = {v: round(ampl.get_value(v), 4) for v in variables}
    solution["profit"] = round(ampl.get_value("Profit"), 4)
    return solution

In [5]:
result, elapsed = solve_charcha()

print("=== CHARCHA WEEKLY PROFIT RESULTS ===")
print(f"Raw material for Input 1 (mp1) -> {result['mp1']:.2f} lbs")
print(f"Raw material for Input 2 (mp2) -> {result['mp2']:.2f} lbs")
print(f"Process 1 runs (Product A)     -> {result['p1']:.2f} oz")
print(f"Process 2 runs (Product B)     -> {result['p2']:.2f} oz")
print(f"Product C produced             -> {result['prodC']:.2f} oz")
print(f"Product D produced             -> {result['prodD']:.2f} oz")
print(f"Waste dumped in river          -> {result['waste_river']:.2f} oz")
print(f"Maximum weekly profit          -> ${result['profit']:.2f}")
print(f"Time                           -> {elapsed:.8f} seconds")

HiGHS 1.11.0=== CHARCHA WEEKLY PROFIT RESULTS ===
Raw material for Input 1 (mp1) -> 1356.44 lbs
Raw material for Input 2 (mp2) -> 386.14 lbs
Process 1 runs (Product A)     -> 1158.42 oz
Process 2 runs (Product B)     -> 0.00 oz
Product C produced             -> 198.02 oz
Product D produced             -> 0.00 oz
Waste dumped in river          -> 1000.00 oz
Maximum weekly profit          -> $6366.34
Time                           -> 0.01085278 seconds


## Sensitivity Analysis

In [6]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var mp1 >= 0;
    var mp2 >= 0;
    var p1 >= 0;
    var p2 >= 0;
    var prodC >= 0;
    var prodD >= 0;
    var waste_river >= 0;

    maximize Profit:
        18 * p1 + 24 * p2 + 11 * prodC + 7 * prodD
        - 6 * (mp1 + mp2)
        - 2 * mp1 - 4 * mp2
        - 1 * p1 - 8 * p2
        - 4 * prodC - 5 * prodD;

    subject to Input1Balance:
        2 * mp1 >= 2 * p1 + 1 * p2 + 2 * prodC;

    subject to Input2Balance:
        3 * mp2 >= 1 * p1 + 2 * p2 + 2 * prodD;

    subject to WasteBalance:
        1 * p1 + 0.8 * p2 = waste_river + 0.8 * prodC + 1.2 * prodD;

    subject to RiverLimit:
        waste_river <= 1000;

    subject to DemandA:
        p1 <= 5000;

    subject to DemandB:
        p2 <= 5000;

    subject to ProcessingTime:
        2 * mp1 + 2 * mp2 + 2 * p1 + 3 * p2 + 1 * prodC + 1 * prodD <= 6000;
    '''
)
ampl.solve()

constraints = ["Input1Balance", "Input2Balance", "WasteBalance",
               "RiverLimit", "DemandA", "DemandB", "ProcessingTime"]

print("=== SHADOW PRICES AND SLACK ===")
print(f"{'Constraint':<20} {'Shadow Price':>14} {'Slack':>10}")
print("-" * 46)
for c in constraints:
    con = ampl.get_constraint(c)
    print(f"{c:<20} {con.dual():>14.4f} {con.slack():>10.4f}")

print()
print("=== REDUCED COSTS ===")
variables = ["mp1", "mp2", "p1", "p2", "prodC", "prodD", "waste_river"]
for v in variables:
    var = ampl.get_variable(v)
    print(f"{v:<15} RC = {var.rc():.4f}   Value = {var.value():.4f}")

HiGHS 1.11.0=== SHADOW PRICES AND SLACK ===
Constraint             Shadow Price      Slack
----------------------------------------------
Input1Balance               -4.5248     0.0000
Input2Balance               -3.6832    -0.0000
WasteBalance                 3.2178    -0.0000
RiverLimit                   3.2178     0.0000
DemandA                      0.0000  3841.5842
DemandB                      0.0000  5000.0000
ProcessingTime               0.5248    -0.0000

=== REDUCED COSTS ===
mp1             RC = 0.0000   Value = 1356.4356
mp2             RC = -0.0000   Value = 386.1386
p1              RC = 0.0000   Value = 1158.4158
p2              RC = -0.0396   Value = 0.0000
prodC           RC = -0.0000   Value = 198.0198
prodD           RC = -2.0297   Value = 0.0000
waste_river     RC = 0.0000   Value = 1000.0000


## Interpretation

The sensitivity analysis reveals which resources are most valuable to Charcha:

- **Processing Time**: The shadow price indicates the marginal value of one additional hour of processing time. If this constraint is binding, it is the bottleneck of the operation.
- **River Limit**: If binding, the shadow price shows how much profit Charcha gains per additional ounce of permitted river dumping. This has both economic and environmental implications.
- **Demand Caps (A, B)**: If not binding, it means Charcha cannot produce enough to meet the maximum sales potential — the bottleneck is elsewhere.
- **Input Balances**: Shadow prices on input constraints show the marginal value of additional input availability.

Variables with zero reduced cost are in the optimal basis. Non-zero reduced costs indicate how much the objective coefficient would need to improve before that variable enters the solution.